In [1]:
# --- STEP 1: Install dependencies ---
!pip install transformers torch tqdm -q

In [2]:
# --- STEP 2: Imports ---
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from tqdm import tqdm
from google.colab import files

In [3]:
# --- STEP 3: Load FinBERT model ---
MODEL_NAME = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

print("✅ FinBERT model loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: yiyanghkust/finbert-tone
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FinBERT model loaded successfully!


In [5]:
# --- STEP 4: Upload and load financial news dataset ---
uploaded = files.upload()                    # ← File picker appears; select your CSV
news_path = list(uploaded.keys())[0]         # ← Picks up the uploaded filename automatically

df = pd.read_csv(news_path)

print("📄 Dataset loaded! Columns:", df.columns.tolist())
print(df.head())

# Ensure published_date is datetime
df["published_date"] = pd.to_datetime(df["published_date"])

Saving finance_news_dataset.csv to finance_news_dataset.csv
📄 Dataset loaded! Columns: ['Content', 'Summary', 'Sentiment', 'published_date']
                                             Content  \
0  On Monday, European Union foreign ministers re...   
1  Bank of Korea adds two banks to digital won tr...   
2  Nvidia CEO Jensen Huang pointed to a fast-risi...   
3  Iran has scared off most ships from the Strait...   
4  European countries are reluctant to get involv...   

                                             Summary Sentiment  \
0  On Monday, European Union foreign ministers re...  Positive   
1  Bank of Korea adds two banks to digital won tr...  Positive   
2  Nvidia CEO Jensen Huang pointed to a fast-risi...   Neutral   
3  Iran has scared off most ships from the Strait...  Negative   
4  European countries are reluctant to get involv...  Negative   

        published_date  
0  2026-03-18 14:31:25  
1  2026-03-18 14:31:25  
2  2026-03-18 14:31:25  
3  2026-03-18 14:31:25  


In [6]:
# --- STEP 5: Define helper function to compute sentiment impact (-1 to 1) ---
def get_sentiment_score(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        labels = ["negative", "neutral", "positive"]
        score = probs[0, 2].item() - probs[0, 0].item()  # (pos - neg)
    return round(score, 2)

In [7]:
# --- STEP 6: Compute sentiment score for each news content ---
tqdm.pandas(desc="Analyzing News Sentiment")
df["Impact"] = df["Content"].progress_apply(get_sentiment_score)

print("\n✅ Sample sentiment scores:")
print(df[["Content", "Impact"]].head())

Analyzing News Sentiment: 100%|██████████| 38582/38582 [1:43:04<00:00,  6.24it/s]


✅ Sample sentiment scores:
                                             Content  Impact
0  On Monday, European Union foreign ministers re...   -0.62
1  Bank of Korea adds two banks to digital won tr...   -1.00
2  Nvidia CEO Jensen Huang pointed to a fast-risi...   -0.17
3  Iran has scared off most ships from the Strait...    1.00
4  European countries are reluctant to get involv...    0.21


In [8]:
# --- STEP 7: Aggregate average impact per date ---
impact_df = df.groupby(df["published_date"].dt.date)["Impact"].mean().reset_index()
impact_df.rename(columns={"published_date": "Date"}, inplace=True)
impact_df["Impact"] = impact_df["Impact"].round(2)

print("\n✅ Final aggregated dataset preview:")
print(impact_df.head())


✅ Final aggregated dataset preview:
         Date  Impact
0  2018-01-01   -0.59
1  2018-01-02   -0.54
2  2018-01-03   -0.56
3  2018-01-04   -0.60
4  2018-01-05   -0.55


In [9]:
# --- STEP 8: Save final impact dataset ---
output_path = "/content/news_impact_score.csv"
impact_df.to_csv(output_path, index=False)
print(f"\n💾 Saved daily impact dataset to: {output_path}")


💾 Saved daily impact dataset to: /content/news_impact_score.csv


In [ ]:
# --- STEP 9: Auto-download the CSV to your system ---
files.download(output_path)           

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>